# Incremental Silver Processing
Incremental silver layer processing using only new bronze rows

## 🔧 Step 1: Import & Setup

> **What this cell does:**
> - Imports Spark, Window functions, and Delta helpers
> - Switches to the target catalog
> - Ensures the **silver** schema exists
> - Generates a unique `silver_run_id` for the current run

In [0]:
# Import Spark functions, Window, Delta and utility libraries
from pyspark.sql import functions as f
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
# Set catalog, create silver schema and generate a unique id for this run
spark.sql("use catalog novacartdbx")
spark.sql("create schema if not exists silver_schema")

silver_run_id = str(uuid.uuid4())

print("Current Silver run is:", silver_run_id)

## 🗂️ Step 2: Create Silver Control Table

> **What this cell does:**
> - Creates a **silver control table** that stores the latest processing state of each entity
> - Helps us track:
>   - Latest **bronze batch** already processed by silver
>   - Latest **bronze processed timestamp**
>   - How many **rows were processed** in the latest silver run

In [0]:
# Control table that stores the last processed bronze watermark per entity
spark.sql("""
          create table if not exists novacartdbx.silver_schema.processing_control(
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingested_at timestamp,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp
          )
          using delta
          """)

   
## 🛠️ Step 3: Create Helper Functions

> **What this cell does:**
> - Defines **4 reusable helper functions** to power the incremental silver pipeline:
>
> | Function | Purpose |
> |---|---|
> | `upsert_to_silver` | Merges cleaned/transformed rows into the **silver target table** |
> | `get_last_processed_bronze_ingested_at` | Reads the **silver watermark** (last processed bronze timestamp) |
> | `upsert_silver_control` | Updates the **silver control table** with the latest processing state |
> | `get_incremental_bronze` | Reads only **new bronze rows** that silver has not yet processed |

In [0]:
# Merge rows into target table by key, or create the table on first run
def upsert_to_silver(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (
            dt.alias("target")
            .merge(df_source.alias("source"),f"target.{join_key} = source.{join_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        df_source.write.format("delta").saveAsTable(target_table)


In [0]:
# Get the latest successful bronze_ingested_at watermark for an entity
def get_last_processed_bronze_ingested_at(entity_name: str):
    ctrl = (
        spark.table("novacartdbx.silver_schema.processing_control")
        .filter((f.col("layer") == 'silver') &
                (f.col("entity_name") == entity_name) &
                (f.col("run_status") == 'success')
                )
        .orderBy(f.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None

    return rows[0]["last_processed_bronze_ingested_at"] 

In [0]:
# Save the new watermark, row count and run status into the control table
def upsert_silver_control(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged):
    ctrl_df = spark.createDataFrame([
        (
            "silver",
            entity_name,
            last_processed_bronze_run_id,
            last_processed_bronze_ingested_at,
            int(rows_merged),
            "success",
            silver_run_id,
            datetime.now()
        )],
            schema="""
                layer string,
                entity_name string,
                last_processed_bronze_run_id string,
                last_processed_bronze_ingested_at timestamp,
                rows_merged bigint,
                run_status string,
                silver_run_id string,
                updated_at timestamp
            """
    )

    dt = DeltaTable.forName(spark, "novacartdbx.silver_schema.processing_control")
    (
        dt.alias("t")
        .merge(ctrl_df.alias("s"),f"t.layer = s.layer AND t.entity_name = s.entity_name")
        .whenMatchedUpdate(set = {
            "last_processed_bronze_run_id":"s.last_processed_bronze_run_id",
            "last_processed_bronze_ingested_at":"s.last_processed_bronze_ingested_at",
            "rows_merged":"s.rows_merged",
            "run_status":"s.run_status",
            "silver_run_id":"s.silver_run_id",
            "updated_at":"s.updated_at"
        }).whenNotMatchedInsertAll()
        .execute()
        )

In [0]:
# Return only bronze rows newer than the watermark (all rows on first run)
def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df, last_ingested_at
    
    return bronze_df.filter(f.col("bronze_ingested_at") > f.lit(last_ingested_at)), last_ingested_at


## 🛒 Step 4: Orders — Incremental Processing

> **What this cell does:**
> 1. **Read** only new bronze rows (incremental)
> 2. **Clean** values like `order_status` and `order_amount`
> 3. **Deduplicate** — keep only the latest version per `order_id`
> 4. **Validate** business rules
> 5. **Send** bad rows to quarantine
> 6. **Merge** good rows into `orders_transformed`

In [0]:
# Preview raw bronze orders
df_raw = spark.sql("select * from novacartdbx.bronze_schema.orders_raw")
display(df_raw)

In [0]:
# Read only new bronze orders since the last silver run
orders_inc, last_ingested_at = get_incremental_bronze("novacartdbx.bronze_schema.orders_raw", "orders")

orders_inc_count = orders_inc.count()
print(f"orders row_to_process_in_silver: {orders_inc_count}")



if orders_inc_count > 0:
    # Window to keep the latest version of each order_id
    order_window = Window.partitionBy("order_id").orderBy(f.col("updated_at").cast("timestamp").desc(),
                                                          f.col("bronze_ingested_at").desc())
    
    # Clean status and amount, cast timestamps and remove duplicates
    orders_cleaned = (
        orders_inc.withColumn("order_status", f.upper(f.trim(f.col("order_status"))))
        .withColumn("order_status",f.when(f.col("order_status") == "", f.lit(None)).otherwise(f.col("order_status")))
        .withColumn("order_amount",f.regexp_replace(f.col("order_amount"), r"[$, ]",""))
        .withColumn("order_amount", f.when(f.trim(f.col("order_amount")).isin("N/A","NULL","??",""), None).otherwise(f.col("order_amount")))
        .withColumn("order_amount",f.col("order_amount").cast("double"))
        .withColumn("created_at", f.to_timestamp("created_at"))
        .withColumn("updated_at", f.to_timestamp("updated_at"))
        .withColumn("row_rank", f.row_number().over(order_window))
        .filter(f.col("row_rank")==1)
        .drop("row_rank")
        .withColumn("silver_run_id", f.lit(silver_run_id))   
    )

    # Merge cleaned orders into orders_cleaned
    upsert_to_silver(
        orders_cleaned,
        "novacartdbx.silver_schema.orders_cleaned",
        "order_id"
    )


    # Flag business rule issues and add date parts
    orders_validated = (orders_cleaned.withColumn(
        "to_be_verified_by_orders_team",
        f.when(f.col("customer_id").isNull(), "verify_customer_id")
        .when(f.col("product_id").isNull(), "verify_product_id")
        .when(f.col("order_status").isNull() | (f.trim(f.col("order_status")) == ""), "verify_order_status")
        .when(f.col("order_amount").isNull() | (f.col("order_amount") <= 0),"verify_order_amount")
        .otherwise("No issues")
        ).withColumn(
            "check_order_amount",
            f.when(f.col("order_amount").isNull() | (f.col("order_amount") <= 0), f.lit(True))
            .otherwise(f.lit(False))
        ).withColumn("order_date", f.to_date("created_at"))
        .withColumn("order_year", f.year("created_at"))
        .withColumn("order_month", f.month("created_at"))
        .withColumn("order_day", f.dayofmonth("created_at"))
        .withColumn("order_dow", f.date_format("created_at", "E"))
    )


    # Split rows into good and bad (quarantine)
    orders_good = orders_validated.filter(f.col("to_be_verified_by_orders_team") == "No issues")

    orders_bad = (
        orders_validated.filter(f.col("to_be_verified_by_orders_team") != "No issues")
        .withColumn("quarantine_ts", f.current_timestamp())
    )

    # Merge good orders into orders_transformed
    upsert_to_silver(
        orders_good, 
        "novacartdbx.silver_schema.orders_transformed",
        "order_id"
    )

    # Append bad orders to quarantine table
    orders_bad.write.format("delta").mode("append").saveAsTable("novacartdbx.silver_schema.orders_quarantine")
    
    # Capture the new watermark and update control table
    mx_ingested = orders_inc.agg(f.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        orders_inc.filter(f.col("bronze_ingested_at") == f.lit(mx_ingested))
        .agg(f.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )
    
    upsert_silver_control("orders", mx_run, mx_ingested, orders_good.count())

else:
    # No new rows: keep the previous watermark
    print("NO NEW orders for bronze rows for silver")

    upsert_silver_control(
        "orders",
        None,
        last_ingested_at,
        orders_inc_count
    )    


   
## 🛍️ Step 5: Products — Incremental Processing

> **What this cell does:**
> 1. **Read** only new bronze rows (incremental)
> 2. **Clean** product names (trim, normalize casing)
> 3. **Standardise** category values
> 4. **Parse** prices — clean symbols and convert to numeric
> 5. **Deduplicate** — keep only the latest record per `product_id`
> 6. **Validate** data quality rules
> 7. **Send** bad rows to quarantine
> 8. **Merge** good rows into the silver current-state table

In [0]:
# Read only new bronze products since the last silver run
product_inc, last_products_ingested_at = get_incremental_bronze("novacartdbx.bronze_schema.products_raw", "products")

product_inc_count = product_inc.count()
print(f"products row_to_process_in_silver: {product_inc_count}")

if product_inc_count > 0:
    # Window to keep the latest version of each product_id
    product_window = (Window.partitionBy("product_id")
    .orderBy(f.col("updated_at").cast("timestamp").desc(),
             f.col("bronze_ingested_at").desc()))
    
    # Clean name, category and price, then remove duplicates
    products_cleaned = (
        product_inc.withColumn("product_name", f.upper(f.trim(f.col("product_name"))))
        # Standardise variants like PROD_9, PRODUCT-11, '  Product 15  ' to 'PRODUCT <n>'
        .withColumn("product_name", f.regexp_replace(f.col("product_name"), r"^PROD(UCT)?[\s_\-]*(\d+)$", "PRODUCT $2"))
        .withColumn("product_name",f.when(f.col("product_name") == "", f.lit(None)).otherwise(f.col("product_name")))
        .withColumn("category", f.when(f.upper(f.trim(f.col("category"))).contains("ELECTRNICS"), "ELECTRONICS")
                    .otherwise(f.upper(f.trim(f.col("category"))))
                    )
        .withColumn("price",f.trim(f.col("price")))
        .withColumn("price", f.regexp_replace(f.col("price"), r"\$", ""))
        .withColumn("price", f.regexp_replace(f.col("price"), ",","."))
        .withColumn("price", f.regexp_replace(f.col("price"), r"\s+", ""))
        .withColumn("price", f.expr("try_cast(price as double)"))
        .withColumn("updated_at", f.to_timestamp("updated_at"))

        .withColumn("row_rank", f.row_number().over(product_window))
        .filter(f.col("row_rank")==1)
        .drop("row_rank")
        .withColumn("silver_run_id", f.lit(silver_run_id))
    )

    # Merge cleaned products into products_cleaned
    upsert_to_silver(
        products_cleaned,"novacartdbx.silver_schema.products_cleaned",
        "product_id"
    )

    # Flag data quality issues (name, category, price)
    products_validated = (
        products_cleaned.withColumn(
            "to_be_verified_by_products_team",
            f.when(f.col("product_name").isNull(), "verify_product_name")
            .when(f.col("category").isNull(), "verify_category")
            .when(f.col("price").isNull() | (f.col("price") <= 0),"verify_price")
            .otherwise("No Issues"))
        .withColumn(
            "check_product_price",
            f.when(f.col("price").isNull() | (f.col("price") <= 0), "invalid_price")
            .otherwise("valid_price"))
    )

    # Keep only valid products
    products_good = products_validated.filter((f.col("to_be_verified_by_products_team") == "No Issues") &
                                              (f.col("check_product_price") == "valid_price")
    )

    # Drop helper column if it exists
    if "price_raw" in products_good.columns:
        products_good = products_good.drop("price_raw")


    # Invalid products go to quarantine
    products_bad = products_validated.filter((f.col("to_be_verified_by_products_team") != "No Issues") | 
                                  (f.col("check_product_price") == "invalid_price")
                                  ).withColumn("quarantine_ts", f.current_timestamp())
                                  
    # Merge good products into products_transformed
    upsert_to_silver(
        products_good, 
        "novacartdbx.silver_schema.products_transformed",
        "product_id"
    )

    # Append bad products to quarantine table
    products_bad.write.format("delta").mode("append").saveAsTable("novacartdbx.silver_schema.products_quarantine")

    # Capture the new watermark and update control table
    mx_ingested = product_inc.agg(f.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        product_inc.filter(f.col("bronze_ingested_at") == f.lit(mx_ingested))
        .agg(f.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )
    upsert_silver_control("products", mx_run, mx_ingested, products_good.count())


else:
    # No new rows: keep the previous watermark
    print("No new product Bronze rows for Silver") 
    upsert_silver_control(
        "products",
        None,
        last_products_ingested_at,
        product_inc_count
    )       

In [0]:
%sql
-- Preview cleaned products in silver
select * from novacartdbx.silver_schema.products_cleaned

## Step 6: Payments — Incremental Processing

> **What this cell does:**
> 1. **Read** only new bronze rows (incremental)
> 2. **Clean** `payment_status` (trim, uppercase) and `paid_amount` (strip `$`, fix decimal commas, cast to double)
> 3. **Deduplicate** — keep only the latest record per `payment_id`
> 4. **Validate** business rules (order id, status, amount)
> 5. **Send** bad rows to quarantine
> 6. **Merge** good rows into `payments_transformed`

In [0]:
# Read only new bronze payments since the last silver run
payments_inc, last_payments_ingested_at = get_incremental_bronze("novacartdbx.bronze_schema.payments_raw", "payments")

payments_inc_count = payments_inc.count()
print(f"payments row_to_process_in_silver: {payments_inc_count}")

if payments_inc_count > 0:
    # Window to keep the latest version of each payment_id
    payment_window = (Window.partitionBy("payment_id")
                      .orderBy(f.col("processed_at").cast("timestamp").desc(),
                               f.col("bronze_ingested_at").desc()))

    # Clean status and amount, cast timestamp and remove duplicates
    payments_cleaned = (
        payments_inc.withColumn("payment_status", f.upper(f.trim(f.col("payment_status"))))
        .withColumn("payment_status", f.when(f.col("payment_status") == "", f.lit(None)).otherwise(f.col("payment_status")))
        .withColumn("paid_amount", f.trim(f.col("paid_amount")))
        .withColumn("paid_amount", f.regexp_replace(f.col("paid_amount"), r"\$", ""))
        .withColumn("paid_amount", f.regexp_replace(f.col("paid_amount"), ",", "."))
        .withColumn("paid_amount", f.regexp_replace(f.col("paid_amount"), r"\s+", ""))
        .withColumn("paid_amount", f.expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at", f.to_timestamp("processed_at"))
        .withColumn("row_rank", f.row_number().over(payment_window))
        .filter(f.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", f.lit(silver_run_id))
    )

    # Merge cleaned payments into payments_cleaned
    upsert_to_silver(
        payments_cleaned,
        "novacartdbx.silver_schema.payments_cleaned",
        "payment_id"
    )

    # Flag business rule issues (order id, status, amount)
    payments_validated = (
        payments_cleaned.withColumn(
            "to_be_verified_by_payments_team",
            f.when(f.col("order_id").isNull(), "verify_order_id")
            .when(f.col("payment_status").isNull(), "verify_payment_status")
            .when(f.col("paid_amount").isNull() | (f.col("paid_amount") <= 0), "verify_paid_amount")
            .otherwise("No Issues"))
        .withColumn("check_paid_amount", f.when(f.col("paid_amount").isNull() | (f.col("paid_amount") <= 0), 
                                                f.lit(True)).otherwise(f.lit(False))
        )
    )

    # Split rows into good and bad (quarantine)
    payments_good = payments_validated.filter(f.col("to_be_verified_by_payments_team") == "No Issues")

    payments_bad = (
        payments_validated.filter(f.col("to_be_verified_by_payments_team") != "No Issues")
        .withColumn("quarantine_ts", f.current_timestamp())
    )

    # Merge good payments into payments_transformed
    upsert_to_silver(
        payments_good,
        "novacartdbx.silver_schema.payments_transformed",
        "payment_id"
    )

    # Append bad payments to quarantine table
    payments_bad.write.format("delta").mode("append").saveAsTable("novacartdbx.silver_schema.payments_quarantine")

    # Capture the new watermark and update control table
    mx_ingested = payments_inc.agg(f.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        payments_inc.filter(f.col("bronze_ingested_at") == f.lit(mx_ingested))
        .agg(f.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )
    upsert_silver_control("payments", mx_run, mx_ingested, payments_good.count())

else:
    # No new rows: keep the previous watermark
    print("No new payment Bronze rows for Silver")
    upsert_silver_control(
        "payments",
        None,
        last_payments_ingested_at,
        payments_inc_count
    )

In [0]:
%sql
-- Preview cleaned payments in silver
select * from novacartdbx.silver_schema.payments_cleaned

   
## ✅ Step 7: Quick Validation

> **What this cell does:**
> - Prints the **row count** for each silver transformed table (`orders`, `products`, `payments`)
> - Displays the **silver control table** so you can confirm incremental processing behaviour
> - Helps verify that upserts, quarantine routing, and watermark tracking all worked correctly

In [0]:
# Show row counts of silver tables and the control table state
print("orders_transformed:", spark.table("novacartdbx.silver_schema.orders_transformed").count())
print("products_transformed:", spark.table("novacartdbx.silver_schema.products_transformed").count())
print("payments_transformed:", spark.table("novacartdbx.silver_schema.payments_transformed").count())
display(spark.table("novacartdbx.silver_schema.processing_control"))